In [ ]:
import pandas as pd
import subprocess
import os

In [ ]:
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

%env USER_NAME={USER_NAME}

# Plink array summary statistics

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m ls -lah gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1/

## Script to compute sumstats

Since this is a dsub job, we need the input files to be copied to the container and the input files out of the container. So they are specified within the dsub jobs

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script takes in the plink array on AOU (non imputed)
# and generates summary statistics to decide appropriate
# QC metrics like threshold for missingness

set -o errexit
set -o nounset

plink1.9 --bfile "${plinkarray_path}/arrays" \
    --freq \
    --missing \
    --hardy \
    --het \
    --memory 14000 \
    --out "${outpath}/plinkarray_sumstats"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats.sh

## Submit dsub job

This script makes a nice folder structure within GCP and does the job over there. The next cell outputs a python env variable called job_id

In [ ]:
!gsutil rm -r ${WORKSPACE_BUCKET}/data/logs/plinkarray-sumstats

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

# n4-highcpu-8 has 8 vCPU and 16 GB RAM

aou_dsub \
    --image biocontainers/plink1.9:v1.90b6.6-181012-1-deb_cv1 \
    --min-ram 16 \
    --min-cores 8 \
    --input-recursive plinkarray_path="gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1" \
    --output-recursive outpath="${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats" \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}.log" \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*' \
    --full

## Read log files

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/logs/plinkarray-sumstats/${JOB_ID}.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/logs/plinkarray-sumstats/${JOB_ID}-stdout.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/logs/plinkarray-sumstats/${JOB_ID}-stderr.log"

#  Plink array summary statistics for subgroups

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m ls -lah gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1/

## Get keep files for ancestries in AoU

In [ ]:
cur_dir = '/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta'

In [ ]:
os.makedirs(f'{cur_dir}',
            exist_ok = True)

ancestry_file = 'gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv'

os.system(f"gsutil -u $GOOGLE_PROJECT -m cp {ancestry_file} {cur_dir}/")

In [ ]:
ancestry_pred = pd.read_csv(f"{cur_dir}/ancestry_preds.tsv", sep='\t')

ancestry_pred.head()

In [ ]:
def getKeepFile(df,
                ancestry,
                ancestry_col = 'ancestry_pred_other',
                id_col = 'research_id',
                save_dir = f'{cur_dir}',
                save_to_ws = True):
    
    df_short = df[df[ancestry_col] == ancestry]
    
    # Make plink keep file. FID is zero
    plink_keep = pd.DataFrame({
        'FID': 0,
        'IID': df_short['research_id']
    })
    
    plink_keep.to_csv(f'{save_dir}/plink_array_{ancestry}.keep', sep = '\t', index = False)
    
    if save_to_ws:
        my_bucket = os.getenv('WORKSPACE_BUCKET')

        args = ["gsutil", "cp",
                f"{save_dir}/plink_array_{ancestry}.keep",
                f"{my_bucket}/data/meta/plink_array/sumstats_{ancestry}/"]
        output = subprocess.run(args, capture_output=True)

In [ ]:
for anc in ancestry_pred['ancestry_pred_other'].drop_duplicates():
    getKeepFile(ancestry_pred, ancestry = anc)

## Make parameter file to run parallel jobs

In [ ]:
# Initialize an Empty DataFrame
param = []

my_bucket = os.getenv('WORKSPACE_BUCKET')

for anc in ancestry_pred['ancestry_pred_other'].drop_duplicates():
    param.append({'--input keep_file' : f"{my_bucket}/data/meta/plink_array/sumstats_{anc}/plink_array_{anc}.keep",
                  '--output-recursive outpath' : f"{my_bucket}/data/meta/plink_array/sumstats_{anc}",
                  '--env ancestry' : anc})

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME = f'{cur_dir}/plinkarray_sumstats_bygroupPARAM.tsv'

param.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

%env PARAMETER_FILENAME={PARAMETER_FILENAME}

## Script to compute sumstats

Since this is a dsub job, we need the input files to be copied to the container and the input files out of the container. So they are specified within the dsub jobs

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script takes in the plink array on AOU (non imputed)
# and generates summary statistics to decide appropriate
# QC metrics like threshold for missingness

set -o errexit
set -o nounset

plink1.9 --bfile "${plinkarray_path}/arrays" \
    --keep "${keep_file}" \
    --maf 0.00000001 \
    --freq \
    --missing \
    --hardy \
    --het \
    --memory 9000 \
    --out "${outpath}/plinkarray_sumstats_${ancestry}"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats_bygroup.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats_bygroup.sh

## Submit dsub job

This script makes a nice folder structure within GCP and does the job over there. The next cell outputs a python env variable called job_id

In [ ]:
#!gsutil rm -r ${WORKSPACE_BUCKET}/data/logs/plinkarray-sumstats-bygroup

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image biocontainers/plink1.9:v1.90b6.6-181012-1-deb_cv1 \
    --min-ram 10 \
    --min-cores 1 \
    --input-recursive plinkarray_path="gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1" \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}.log" \
    --tasks "${PARAMETER_FILENAME}" \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_sumstats_bygroup.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

## Read log files

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_eur/plinkarray_sumstats_eur.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_afr/plinkarray_sumstats_afr.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_amr/plinkarray_sumstats_amr.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_eas/plinkarray_sumstats_eas.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_mid/plinkarray_sumstats_mid.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_sas/plinkarray_sumstats_sas.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/sumstats_oth/plinkarray_sumstats_oth.log"